<a href="https://colab.research.google.com/github/salmanmfa2/ddac2026-behh-army/blob/main/Draft_Notebook_DDAC_behh_army.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Behh Army - DDAC AInnovation | IKPA Satker 2021-2026

In [43]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.cluster import KMeans
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

---

In [44]:
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
repo_id = "salmanmfa2/ikpa-tahunan"
years = [2021, 2022, 2023, 2024, 2025, 2026]

print("Memuat file tahunan...")
for year in years:
    try:
        file_path = f"Data Gabungan (2021 s.d. 04-2026)/IKPA_SATKER_{year}.csv"
        local_path = hf_hub_download(repo_id=repo_id, filename=file_path, repo_type="dataset", token=hf_token)

        # Muat ke dalam variabel individu (ikpa_2021, dsb.)
        df_name = f"ikpa_{year}"
        globals()[df_name] = pd.read_csv(local_path, sep=";", on_bad_lines='skip')
        globals()[df_name]['YEAR_SOURCE'] = year # Tambahkan tag tahun untuk pelacakan

        print(f"Berhasil memuat {df_name} dengan {len(globals()[df_name])} baris.")
    except Exception as e:
        print(f"Gagal memuat data tahun {year}. Error: {e}")

# Gabungkan menjadi satu DataFrame master
all_dfs = [globals()[f"ikpa_{y}"] for y in years if f"ikpa_{y}" in globals()]
ikpa_all = pd.concat(all_dfs, axis=0, ignore_index=True, sort=False)
print(f"\nDataFrame Master Terpadu 'ikpa_all' berhasil dibuat dengan total {len(ikpa_all)} baris.")
display(ikpa_all.head())

Memuat file tahunan...
Berhasil memuat ikpa_2021 dengan 235756 baris.
Berhasil memuat ikpa_2022 dengan 232848 baris.
Berhasil memuat ikpa_2023 dengan 227304 baris.
Berhasil memuat ikpa_2024 dengan 228348 baris.
Berhasil memuat ikpa_2025 dengan 222036 baris.
Berhasil memuat ikpa_2026 dengan 220488 baris.

DataFrame Master Terpadu 'ikpa_all' berhasil dibuat dengan total 1366780 baris.


,KDBA,KDKPPN,KDSATKER,NMSATKER,PERIODE,NILAI_REV_DIPA,BOBOT_REV_DIPA,NILAI_HAL3_DIPA,BOBOT_HAL3_DIPA,NILAI_PAGU_MINUS,...,BOBOT_RETUR,NILAI_RENKAS,BOBOT_RENKAS,NILAI_SALAH_SPM,BOBOT_SALAH_SPM,NILAI_TOTAL,KONV_BOBOT,NILAI_AKHIR,YEAR_SOURCE,KDKANWIL
0,6.0,1.0,6050.0,KEJAKSAAN TINGGI ACEH,5,100.0,5.0,81.65,5.0,95.93,...,5.0,0.0,0.0,80.0,5.0,63.90,75.0,85.20,2021,NaN
1,6.0,1.0,6071.0,KEJAKSAAN NEGERI SABANG,2,100.0,5.0,0.00,0.0,100.00,...,5.0,0.0,0.0,80.0,5.0,48.67,53.0,91.83,2021,NaN
2,6.0,1.0,6142.0,CABANG KEJAKSAAN NEGERI PIDIE DI KOTA BAKTI,1,100.0,5.0,0.00,0.0,92.63,...,5.0,0.0,0.0,100.0,5.0,43.93,45.0,97.62,2021,NaN
3,6.0,1.0,6142.0,CABANG KEJAKSAAN NEGERI PIDIE DI KOTA BAKTI,12,100.0,5.0,82.27,5.0,92.63,...,5.0,0.0,0.0,80.0,5.0,68.30,75.0,91.07,2021,NaN
4,54.0,1.0,19674.0,BADAN PUSAT STATISTIK KAB. PIDIE,1,100.0,5.0,0.00,0.0,100.00,...,5.0,0.0,0.0,80.0,5.0,43.01,45.0,95.58,2021,NaN


In [45]:
print("Dimensi DataFrame:", ikpa_all.shape)
print("\nRingkasan Statistik:")
display(ikpa_all.describe().round(3))
print("\nTipe Data Kolom:")
print(ikpa_all.dtypes)

Dimensi DataFrame: (1366780, 36)

Ringkasan Statistik:


,KDBA,KDKPPN,KDSATKER,PERIODE,NILAI_REV_DIPA,BOBOT_REV_DIPA,NILAI_HAL3_DIPA,BOBOT_HAL3_DIPA,NILAI_PAGU_MINUS,BOBOT_PAGU_MINUS,...,BOBOT_RETUR,NILAI_RENKAS,BOBOT_RENKAS,NILAI_SALAH_SPM,BOBOT_SALAH_SPM,NILAI_TOTAL,KONV_BOBOT,NILAI_AKHIR,YEAR_SOURCE,KDKANWIL
count,1366768.000,1366768.000,1366768.000,1366780.000,1366780.000,1363873.000,1366780.000,1364549.000,235744.000,233053.0,...,233053.0,235756.000,235745.000,235756.000,233053.0,1363873.000,1363873.000,1363873.000,1366780.000,670872.000
mean,39.508,73.228,473486.087,6.500,99.802,9.146,71.370,11.385,97.101,5.0,...,5.0,3.882,0.199,89.201,5.0,73.965,86.084,85.614,2023.461,15.528
std,54.118,52.969,191846.602,3.452,2.680,1.882,28.725,4.230,12.267,0.0,...,0.0,19.183,0.977,8.512,0.0,18.653,13.218,16.096,1.708,9.121
min,1.000,1.000,17.000,1.000,0.000,5.000,0.000,0.000,-4.450,5.0,...,5.0,0.000,0.000,0.000,5.0,5.000,25.000,7.140,2021.000,1.000
25%,15.000,29.000,402008.000,3.000,100.000,10.000,56.500,10.000,99.980,5.0,...,5.0,0.000,0.000,80.000,5.0,63.000,80.000,80.590,2022.000,9.000
50%,25.000,60.000,440003.000,6.500,100.000,10.000,78.610,10.000,100.000,5.0,...,5.0,0.000,0.000,90.000,5.0,75.810,80.000,91.790,2023.000,15.000
75%,56.000,120.000,652097.000,10.000,100.000,10.000,97.490,15.000,100.000,5.0,...,5.0,0.000,0.000,100.000,5.0,90.000,100.000,97.000,2025.000,23.000
max,999.000,999.000,999982.000,12.000,100.000,10.000,100.000,15.000,100.000,5.0,...,5.0,100.000,5.000,100.000,5.0,107.500,100.000,107.500,2026.000,34.000



Tipe Data Kolom:
KDBA                    float64
KDKPPN                  float64
KDSATKER                float64
NMSATKER                 object
PERIODE                   int64
NILAI_REV_DIPA          float64
BOBOT_REV_DIPA          float64
NILAI_HAL3_DIPA         float64
BOBOT_HAL3_DIPA         float64
NILAI_PAGU_MINUS        float64
BOBOT_PAGU_MINUS        float64
NILAI_KONTRAKTUAL       float64
BOBOT_KONTRAKTUAL       float64
NILAI_UP_TUP            float64
BOBOT_UP_TUP            float64
NILAI_LPJ               float64
BOBOT_LPJ               float64
NILAI_DISPENSASI_SPM    float64
BOBOT_DISPENSASI_SPM    float64
NILAI_REALISASI         float64
BOBOT_REALISASI         float64
NILAI_TAGIHAN           float64
BOBOT_TAGIHAN           float64
NILAI_CAPUT             float64
BOBOT_CAPUT             float64
NILAI_RETUR             float64
BOBOT_RETUR             float64
NILAI_RENKAS            float64
BOBOT_RENKAS            float64
NILAI_SALAH_SPM         float64
BOBOT_SALAH_SPM       

In [46]:
print(ikpa_all.isnull().sum())

KDBA                         12
KDKPPN                       12
KDSATKER                     12
NMSATKER                     12
PERIODE                       0
NILAI_REV_DIPA                0
BOBOT_REV_DIPA             2907
NILAI_HAL3_DIPA               0
BOBOT_HAL3_DIPA            2231
NILAI_PAGU_MINUS        1131036
BOBOT_PAGU_MINUS        1133727
NILAI_KONTRAKTUAL             0
BOBOT_KONTRAKTUAL           147
NILAI_UP_TUP                  0
BOBOT_UP_TUP                 52
NILAI_LPJ               1131024
BOBOT_LPJ               1131092
NILAI_DISPENSASI_SPM          0
BOBOT_DISPENSASI_SPM       2907
NILAI_REALISASI               0
BOBOT_REALISASI            2907
NILAI_TAGIHAN                 0
BOBOT_TAGIHAN               213
NILAI_CAPUT                   0
BOBOT_CAPUT                2231
NILAI_RETUR             1131024
BOBOT_RETUR             1133727
NILAI_RENKAS            1131024
BOBOT_RENKAS            1131035
NILAI_SALAH_SPM         1131024
BOBOT_SALAH_SPM         1133727
NILAI_TO

In [47]:
print(ikpa_2021.shape)
print(ikpa_2022.shape)
print(ikpa_2023.shape)
print(ikpa_2024.shape)
print(ikpa_2025.shape)
print(ikpa_2026.shape)

(235756, 35)
(232848, 25)
(227304, 25)
(228348, 26)
(222036, 26)
(220488, 26)


### Analisis Konsistensi Skema (Kolom)
Kode di bawah ini akan memetakan semua kolom yang ada di setiap DataFrame (`ikpa_2021` hingga `ikpa_2026`) untuk melihat kolom mana yang bersifat umum dan mana yang spesifik pada tahun tertentu.

In [48]:
import pandas as pd

years = [2021, 2022, 2023, 2024, 2025, 2026]
dfs = {
    2021: ikpa_2021,
    2022: ikpa_2022,
    2023: ikpa_2023,
    2024: ikpa_2024,
    2025: ikpa_2025,
    2026: ikpa_2026
}

# Ambil semua nama kolom unik dari seluruh dataset
all_columns = sorted(list(set().union(*(df.columns for df in dfs.values()))))

# Buat matriks perbandingan
comparison_data = []
for col in all_columns:
    row = {'Nama Kolom': col}
    for year in years:
        row[str(year)] = '✓' if col in dfs[year].columns else ''
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)

print("Tabel Perbandingan Kolom (2021-2026):")
display(df_comparison)

# Identifikasi kolom yang ada di SEMUA tahun
common_cols = [col for col in all_columns if all(col in dfs[yr].columns for yr in years)]
print(f"\nJumlah kolom yang ada di semua tahun: {len(common_cols)}")
print("Kolom bersama:", common_cols)

Tabel Perbandingan Kolom (2021-2026):


,Nama Kolom,2021,2022,2023,2024,2025,2026
0,BOBOT_CAPUT,✓,✓,✓,✓,✓,✓
1,BOBOT_DISPENSASI_SPM,✓,✓,✓,✓,✓,✓
2,BOBOT_HAL3_DIPA,✓,✓,✓,✓,✓,✓
3,BOBOT_KONTRAKTUAL,✓,✓,✓,✓,✓,✓
4,BOBOT_LPJ,✓,,,,,
5,BOBOT_PAGU_MINUS,✓,,,,,
6,BOBOT_REALISASI,✓,✓,✓,✓,✓,✓
7,BOBOT_RENKAS,✓,,,,,
8,BOBOT_RETUR,✓,,,,,
9,BOBOT_REV_DIPA,✓,✓,✓,✓,✓,✓



Jumlah kolom yang ada di semua tahun: 25
Kolom bersama: ['BOBOT_CAPUT', 'BOBOT_DISPENSASI_SPM', 'BOBOT_HAL3_DIPA', 'BOBOT_KONTRAKTUAL', 'BOBOT_REALISASI', 'BOBOT_REV_DIPA', 'BOBOT_TAGIHAN', 'BOBOT_UP_TUP', 'KDBA', 'KDKPPN', 'KDSATKER', 'KONV_BOBOT', 'NILAI_AKHIR', 'NILAI_CAPUT', 'NILAI_DISPENSASI_SPM', 'NILAI_HAL3_DIPA', 'NILAI_KONTRAKTUAL', 'NILAI_REALISASI', 'NILAI_REV_DIPA', 'NILAI_TAGIHAN', 'NILAI_TOTAL', 'NILAI_UP_TUP', 'NMSATKER', 'PERIODE', 'YEAR_SOURCE']


### Verifikasi Struktur ikpa_all
Mari kita lihat berapa banyak kolom yang ada di `ikpa_all` dan bagaimana distribusi nilai kosong (null) pada kolom-kolom yang tidak tersedia di semua tahun.

In [49]:
print(f"Total kolom di ikpa_all: {len(ikpa_all.columns)}")
print("Daftar kolom di ikpa_all:")
print(list(ikpa_all.columns))

# Menunjukkan jumlah baris yang terisi (non-null) untuk kolom-kolom tertentu yang tidak umum
# Contoh: KDKANWIL hanya ada di beberapa tahun
print("\nJumlah data terisi pada kolom non-umum:")
missing_cols = [col for col in ikpa_all.columns if col not in common_cols]
display(ikpa_all[missing_cols].notnull().sum())

Total kolom di ikpa_all: 36
Daftar kolom di ikpa_all:
['KDBA', 'KDKPPN', 'KDSATKER', 'NMSATKER', 'PERIODE', 'NILAI_REV_DIPA', 'BOBOT_REV_DIPA', 'NILAI_HAL3_DIPA', 'BOBOT_HAL3_DIPA', 'NILAI_PAGU_MINUS', 'BOBOT_PAGU_MINUS', 'NILAI_KONTRAKTUAL', 'BOBOT_KONTRAKTUAL', 'NILAI_UP_TUP', 'BOBOT_UP_TUP', 'NILAI_LPJ', 'BOBOT_LPJ', 'NILAI_DISPENSASI_SPM', 'BOBOT_DISPENSASI_SPM', 'NILAI_REALISASI', 'BOBOT_REALISASI', 'NILAI_TAGIHAN', 'BOBOT_TAGIHAN', 'NILAI_CAPUT', 'BOBOT_CAPUT', 'NILAI_RETUR', 'BOBOT_RETUR', 'NILAI_RENKAS', 'BOBOT_RENKAS', 'NILAI_SALAH_SPM', 'BOBOT_SALAH_SPM', 'NILAI_TOTAL', 'KONV_BOBOT', 'NILAI_AKHIR', 'YEAR_SOURCE', 'KDKANWIL']

Jumlah data terisi pada kolom non-umum:


,0
NILAI_PAGU_MINUS,235744
BOBOT_PAGU_MINUS,233053
NILAI_LPJ,235756
BOBOT_LPJ,235688
NILAI_RETUR,235756
BOBOT_RETUR,233053
NILAI_RENKAS,235756
BOBOT_RENKAS,235745
NILAI_SALAH_SPM,235756
BOBOT_SALAH_SPM,233053
